# FinanceSpanQA+

### Goal
1) Import dataset from PhantonData (from hugging face) and test 50 samples
2) Classification task on 50 samples (question + context) as numerical/temporal/obligation-entitlement
3) Verification task: 
 a.Use open source models(Qwen-3-8b, Gemma-3, FinBert, Deepseekor) to verify if 50 samples (question + context + answer) are supported/contradicted/not_found
 b.Return confidence score (0-1)

## TASK 1: Import Phantom Dataset (ShortQA)

In [ ]:
from datasets import load_dataset

dataset = load_dataset(
    "json",
    data_files="hf://datasets/obayreaps/Gemini_Classification_Phantom/qa_generation_gemini_flash.jsonl"
)

print(f"Successfully loaded: {len(dataset['train'])} samples. ✅")


In [ ]:
#Get all samples that are labeled not hallucinated

notHallu_dataset = dataset["train"].filter(
    lambda x: x["ground_truth_label"] == "not hallucination"
)

print(f"Extracted {len(notHallu_dataset)} samples that are not_hallucinated")

In [ ]:
#Extract first 50 samples to work with
small_test_pool = notHallu_dataset.select(range(50))

#Print out each sample's characteristics: (question, contexst, answer, label)
for i in range(len(small_test_pool)):
    print("Test", i+1)
    print("Query: ", small_test_pool[i]["query"])
    print("Context: ", small_test_pool[i]["context"])
    print("Answer: ",small_test_pool[i]["answer"])
    print("Label: ", small_test_pool[i]["ground_truth_label"])
    print("\n")


## Task 2: Classification using Gemini

### Goal: Label each sample to document-type (numerical / temporal/ obligation-entitlement)

In [ ]:
#Import necessary libraries for google gemini
#pip install dotenv
#pip install google-genai

import os, json,time, random
import dotenv
import time, random
from google import genai
from google.genai import errors as genai_errors
from google.genai import types

In [ ]:
#Load Gemini API Key
dotenv.load_dotenv()
API_KEY = os.getenv("GEMINI_API_KEY")
if not API_KEY:
    raise ValueError("GEMINI_API_KEY not found. Put in .env file")

client = genai.Client(api_key=API_KEY)

In [ ]:
import json, os, time, random
from google import genai
from google.genai import errors as genai_errors
from google.genai import types

CLASSIFY_CONFIG = types.GenerateContentConfig(
    response_mime_type="application/json",
    temperature=0.0
)

RETRIABLE_5XX = ("500", "502", "503", "504")

def is_retriable_exception(e: Exception) -> bool:
    s = str(e)
    return (
        "429" in s
        or "RESOURCE_EXHAUSTED" in s
        or any(code in s for code in RETRIABLE_5XX)
        or "UNAVAILABLE" in s
        or "DEADLINE_EXCEEDED" in s
    )

def generate_with_backoff(client, model, contents, config=None, max_retries=8):
    last_err = None
    for attempt in range(max_retries):
        try:
            return client.models.generate_content(model=model, contents=contents, config=config)
        except (genai_errors.ClientError, genai_errors.ServerError) as e:
            last_err = e
            if is_retriable_exception(e):
                # exponential backoff + jitter, with a cap
                sleep_s = min(60.0, (2 ** attempt) + random.uniform(0, 0.5))
                time.sleep(sleep_s)
                continue
            raise
    raise RuntimeError(f"Max retries exceeded. Last error: {last_err}")


def build_classification_prompt(query: str, answer: str) -> str:
    return f"""
You are classifying the QUESTION TYPE into exactly one label.

Labels:
- numeric: asks for a number, amount, percentage, price, quantity, count, ratio, or a specific numeric value.
- temporal: asks about time or dates in a time-relation sense (when, deadline, start/end, duration, term length, renewal period, timeline).
- obligation-entitlement: asks about duties/requirements/permissions/rights (must/shall/required to, may/can, is entitled to, is prohibited, allowed, responsible for).

Rules:
- Choose the best single label.
- Output ONLY valid JSON (no markdown).
- confidence must be a float from 0 to 1.
- reason must be 1 short sentence.

Question:
{query}

Answer:
{answer}

Return JSON like:
{{"label":"numeric|temporal|obligation-entitlement","confidence":0.0,"reason":"..."}}
""".strip()


def classify_with_gemini(query: str, answer: str, model: str = "gemini-2.0-flash"):
    prompt = build_classification_prompt(query, answer) 
    resp = generate_with_backoff(client, model=model, contents=prompt, config=CLASSIFY_CONFIG)

    raw_text = resp.text.strip() if resp and resp.text else ""

    try:
        data = json.loads(raw_text)
    except Exception:
        return {"label": None, "confidence": None, "reason": None}

    # 🔧 Normalize: sometimes model returns a list with one object
    if isinstance(data, list):
        if len(data) > 0 and isinstance(data[0], dict):
            data = data[0]
        else:
            return {"label": None, "confidence": None, "reason": None}

    # Ensure dict shape
    if not isinstance(data, dict):
        return {"label": None, "confidence": None, "reason": None}

    # Return label and other important characteristics
    return {
        "label": data.get("label"),
        "confidence": data.get("confidence"),
        "reason": data.get("reason")
    }


In [ ]:
# 4) Run and save
out_path = "outputs/classification_gemini_50.jsonl" #create file path
os.makedirs("outputs", exist_ok=True) #if output folder doesn't exist, create it. Otherwise do nothing

with open(out_path, "w") as f:
    for i, ex in enumerate(notHallu_dataset):
        q = ex["query"]
        a = ex["answer"]
        c = ex["context"]
        gtl = ex["ground_truth_label"]

        #Creates prediction labels based off question and answer
        pred = classify_with_gemini(q, a)

        #Data added per each sample in output file
        row = {
            "idx": i,
            "query": q,
            "answer": a,
            "context": c,
            "ground_truth_label": gtl,
            "label": pred.get("label"),
            "confidence": pred.get("confidence"),
            "reason": pred.get("reason")
        }
        
        f.write(json.dumps(row) + "\n") #writes row into file

        if (i + 1) % 1000 == 0: #pauses every 1000 samples 
            f.flush()
            os.fsync(f.fileno())
            print(f"✅ Saved checkpoint: {i+1} rows -> {out_path}")

print("Saved:", out_path) #After the program finishes writing the file

In [ ]:
#Extract information from the json file

import json

results = [] #variable for results

with open("outputs/classification_gemini_50_with_ctx_ans.jsonl", "r") as f:
    for line in f:
        results.append(json.loads(line))
        
print("Loaded", len(results), "rows")

print(results[0])
print(results[0]["label"])
print(results[5]["label"])
print(results[2001]["label"])

# Task 2.5: Judge the Predicted Labels

In [ ]:
import json, os
import pandas as pd
from google.genai import types

# --- Judge config (JSON output, deterministic) ---
JUDGE_CONFIG = types.GenerateContentConfig(
    response_mime_type="application/json",
    temperature=0.0
)

def build_judge_prompt(query: str, pred_label: str) -> str:
    # If you also stored "answer" in your jsonl, I can upgrade this prompt to use it.
    return f"""
You are a strict evaluator. Decide if the predicted QUESTION TYPE label is correct.

Valid labels:
- numeric: asks for a number, amount, percentage, price, quantity, count, ratio, or a specific numeric value.
- temporal: asks about time or dates in a time-relation sense (when, deadline, start/end, duration, term length, renewal period, timeline).
- obligation-entitlement: asks about duties/requirements/permissions/rights (must/shall/required to, may/can, entitled to, prohibited, allowed, responsible for).

Question:
{query}

Predicted label:
{pred_label}

Return ONLY valid JSON:
{{
  "is_correct": true/false,
  "correct_label": "numeric|temporal|obligation-entitlement",
  "explanation": "one short sentence"
}}
""".strip()

def judge_one(query: str, pred_label: str, model: str = "gemini-3.0") -> dict:
    prompt = build_judge_prompt(query, pred_label)

    # Reuse YOUR existing backoff helper + client
    resp = generate_with_backoff(client, model=model, contents=prompt, config=JUDGE_CONFIG)

    raw_text = resp.text.strip() if resp and resp.text else ""
    try:
        data = json.loads(raw_text)
    except Exception:
        data = {"is_correct": None, "correct_label": None, "explanation": None}

    # Normalize list outputs just in case
    if isinstance(data, list):
        data = data[0] if (len(data) > 0 and isinstance(data[0], dict)) else {"is_correct": None, "correct_label": None, "explanation": None}

    return {
        "is_correct": data.get("is_correct"),
        "correct_label": data.get("correct_label"),
        "explanation": data.get("explanation")
    }

def run_judge_report(
    jsonl_path: str,
    model: str = "gemini-3.0",
    max_samples: int | None = None,
    save_incorrect_csv: str = "outputs/judge_incorrect_samples.csv"
):
    # Load your existing classification jsonl
    rows = []
    with open(jsonl_path, "r") as f:
        for i, line in enumerate(f):
            if max_samples is not None and i >= max_samples:
                break
            rows.append(json.loads(line))

    judged_rows = []
    for i, r in enumerate(rows):
        q = r.get("query", "")
        pred_label = r.get("label")  # <- your jsonl uses "label" (not "pred_label")

        j = judge_one(q, pred_label, model=model)

        judged_rows.append({
            "idx": r.get("idx", i),
            "query": q,
            "ground_truth_label": r.get("ground_truth_label"),
            "pred_label": pred_label,
            "judge_is_correct": j["is_correct"],
            "judge_correct_label": j["correct_label"],
            "judge_explanation": j["explanation"],
        })

        if (i + 1) % 500 == 0:
            print(f"✅ Judged {i+1}/{len(rows)}")

    df = pd.DataFrame(judged_rows)

    # Only evaluate rows where judge returned a boolean
    eval_df = df[df["judge_is_correct"].isin([True, False])].copy()
    total = len(eval_df)
    num_correct = int((eval_df["judge_is_correct"] == True).sum())
    num_incorrect = int((eval_df["judge_is_correct"] == False).sum())
    pct_correct = (num_correct / total * 100.0) if total else 0.0
    pct_incorrect = (num_incorrect / total * 100.0) if total else 0.0

    print("\n===== Gemini-3.0 Judge Report =====")
    print("File:", jsonl_path)
    print("Evaluated:", total)
    print(f"Not hallucinated (correct): {num_correct} ({pct_correct:.2f}%)")
    print(f"Hallucinated / not true (incorrect): {num_incorrect} ({pct_incorrect:.2f}%)")
    print("Not correctly labeled (count):", num_incorrect)

    incorrect_df = eval_df[eval_df["judge_is_correct"] == False][
        ["idx", "query", "ground_truth_label", "pred_label", "judge_correct_label", "judge_explanation"]
    ].copy()

    os.makedirs(os.path.dirname(save_incorrect_csv), exist_ok=True)
    incorrect_df.to_csv(save_incorrect_csv, index=False)
    print("Saved incorrect table:", save_incorrect_csv)

    return {
        "pct_correct": pct_correct,
        "pct_incorrect": pct_incorrect,
        "num_incorrect": num_incorrect,
        "incorrect_df": incorrect_df,
        "full_df": df
    }

# --- Run it ---
report = run_judge_report("outputs/classification_gemini_50.jsonl", model="gemini-2.5-pro", max_samples=None)
report["incorrect_df"].head(25)

In [ ]:
import json
import os

in_path = "outputs/classification_gemini_50_with_ctx_ans.jsonl"
out_path = "outputs/classification_gemini_50_reordered.jsonl"

os.makedirs("outputs", exist_ok=True)

order = [
    "idx",
    "query",
    "context",
    "answer",
    "ground_truth_label",
    "label",
    "confidence",
    "reason",
]

count = 0

with open(in_path, "r") as fin, open(out_path, "w") as fout:
    for line in fin:
        row = json.loads(line)

        new_row = {}

        # First write fields in desired order
        for k in order:
            if k in row:
                new_row[k] = row[k]

        # Then append any leftover fields (just in case)
        for k in row:
            if k not in new_row:
                new_row[k] = row[k]

        fout.write(json.dumps(new_row, ensure_ascii=False) + "\n")
        count += 1

print(f"✅ Safely wrote {count} rows to {out_path}")
print("🛡️ Original file untouched:", in_path)

# Task 3: Verification Task

Procedures
1. Generate answer from both GPT-4o and Gemini-2.0-flask by passing query and context
2. Use Gemini-2.0 as a judge and compare the outputs to the gold answers in output/classification...json file
3. Record # of answers generated that align with original and those that contradict/hallucinate with original

In [ ]:
def generate_qa_with_models(query: str, context: str, claim_type: str | None = None, model: str = "gemini-2.0-flash"):
    prompt = build_typed_qa_prompt(query, context, claim_type=claim_type)

    resp = generate_with_backoff(client, model=model, contents=prompt, config=QA_CONFIG)
    raw_text = resp.text.strip() if resp and resp.text else ""

    # Default return shape
    default = {"claim_type_used": None, "answer": None, "confidence": None, "support": None}

    try:
        data = json.loads(raw_text)
    except Exception:
        return default

    # Normalize list wrapper
    if isinstance(data, list):
        if len(data) > 0 and isinstance(data[0], dict):
            data = data[0]
        else:
            return default

    if not isinstance(data, dict):
        return default

    # Normalize answer string
    ans = data.get("answer")
    if isinstance(ans, str):
        ans = ans.strip()
        if ans.lower() == "abstain":
            ans = "abstain"

    return {
        "claim_type_used": data.get("claim_type_used"),
        "answer": ans,
        "confidence": data.get("confidence"),
        "support": data.get("support"),
    }

In [ ]:
def build_typed_qa_prompt(query: str, context: str, claim_type: str | None = None, max_ctx_chars: int = 6000) -> str:
    ctx = context or ""
    if len(ctx) > max_ctx_chars:
        ctx = ctx[:max_ctx_chars] + "\n[TRUNCATED]"

    ct = (claim_type or "unknown").strip()

    return f"""
You are answering a short-answer question using ONLY the provided context.

Global rules:
- Use ONLY the context. No outside knowledge.
- If the answer is not explicitly stated or not fully supported by the context, output exactly: "abstain".
- Keep the answer short (no extra explanation).

Claim type: {ct}
{_rules_for_claim_type(ct)}

Return ONLY valid JSON:
{{
  "claim_type_used": "numeric|temporal|obligation-entitlement",
  "answer": "abstain OR a short answer string",
  "confidence": 0.0,
  "support": "one short sentence citing what in the context supports the answer OR why abstain"
}}

Context:
{ctx}

Question:
{query}
""".strip()

In [ ]:
def _rules_for_claim_type(claim_type: str) -> str:
    ct = (claim_type or "").strip().lower()

    if ct == "numeric":
        return """Type-specific rules (numeric):
- Return ONLY the numeric value (include units like %, $, years if stated).
- Do NOT compute or infer values unless the context explicitly provides the computed value.
- If multiple candidate values exist and it is ambiguous, output "abstain".
"""
    if ct == "temporal":
        return """Type-specific rules (temporal):
- Return the exact date/time/duration stated in the context.
- If relative time is given but the anchor date is missing, output "abstain".
- If multiple candidate dates exist and it is ambiguous, output "abstain".
"""
    if ct == "obligation-entitlement":
        return """Type-specific rules (obligation-entitlement):
- Answer using explicit obligation/permission language from the context (must/shall/may/prohibited/entitled).
- Do NOT strengthen or weaken modality (must ≠ may).
- If the obligation/permission is not clearly stated, output "abstain".
"""
    return """Type-specific rules (unknown):
- Infer the claim type (numeric/temporal/obligation-entitlement) from the question.
- Then follow the corresponding rules above.
"""

In [ ]:
# 2) Backoff helpers (your exact pattern)
RETRIABLE_5XX = ("500", "502", "503", "504")

def is_retriable_exception(e: Exception) -> bool:
    s = str(e)
    return (
        "429" in s
        or "RESOURCE_EXHAUSTED" in s
        or any(code in s for code in RETRIABLE_5XX)
        or "UNAVAILABLE" in s
        or "DEADLINE_EXCEEDED" in s
    )

def generate_with_backoff(client, model, contents, config=None, max_retries=8):
    last_err = None
    for attempt in range(max_retries):
        try:
            return client.models.generate_content(model=model, contents=contents, config=config)
        except (genai_errors.ClientError, genai_errors.ServerError) as e:
            last_err = e
            if is_retriable_exception(e):
                sleep_s = min(60.0, (2 ** attempt) + random.uniform(0, 0.5))
                time.sleep(sleep_s)
                continue
            raise
    raise RuntimeError(f"Max retries exceeded. Last error: {last_err}")


In [ ]:
import os, time, random
from google import genai
from google.genai import errors as genai_errors


QA_CONFIG = types.GenerateContentConfig(
    response_mime_type="application/json",
    temperature=0.0
)


def generate_ans_gemini(query: str, context: str, claim_type: str | None = None, model: str = "gemini-2.0-flash") -> dict:
    prompt = build_typed_qa_prompt(query, context, claim_type=claim_type)

    resp = generate_with_backoff(client, model=model, contents=prompt, config=QA_CONFIG)
    raw_text = resp.text.strip() if resp and resp.text else ""

    try:
        data = json.loads(raw_text)
    except Exception:
        data = {"claim_type_used": None, "answer": None, "confidence": None, "support": None}

    # normalize list -> dict edge case
    if isinstance(data, list):
        data = data[0] if (len(data) > 0 and isinstance(data[0], dict)) else {"claim_type_used": None, "answer": None, "confidence": None, "support": None}

    # keep only expected fields
    return {
        "claim_type_used": data.get("claim_type_used"),
        "answer": data.get("answer"),
        "confidence": data.get("confidence"),
        "support": data.get("support"),
    }


## Gemini Verifcation Call Block

In [ ]:
# 1) Gemini client (needs env var set)
dotenv.load_dotenv(dotenv_path="PhantomDataset/.env", override=True)
API_KEY = os.getenv("GEMINI_API_KEY")
if not API_KEY:
    raise ValueError("Missing GEMINI_API_KEY in environment.")
client = genai.Client(api_key=API_KEY)

In [ ]:
out_path = "outputs/qa_generation_gemini_flash.jsonl"
os.makedirs("outputs", exist_ok=True)

with open(out_path, "w") as f:
    for i, ex in enumerate(notHallu_dataset):
        q = ex["query"]
        c = ex["context"]
        gold = ex["answer"]                  # ground truth answer
        gtl = ex.get("ground_truth_label")   # your dataset label (if present)

        # If you already have a claim type label stored somewhere, pass it here.
        # If not, leave as None and the model will infer.
        claim_type = ex.get("label")  # if your enriched dataset includes it; otherwise None

        pred = generate_qa_with_models(q, c, claim_type=claim_type, model="gemini-2.0-flash")

        row = {
            "idx": i,
            "query": q,
            "context": c,
            "gold_answer": gold,
            "ground_truth_label": gtl,
            "claim_type_param": claim_type,
            "claim_type_used": pred.get("claim_type_used"),
            "model_answer": pred.get("answer"),
            "model_confidence": pred.get("confidence"),
            "model_support": pred.get("support"),
        }

        f.write(json.dumps(row, ensure_ascii=False) + "\n")

        if (i + 1) % 1000 == 0:
            f.flush()
            os.fsync(f.fileno())
            print(f"✅ Saved checkpoint: {i+1} rows -> {out_path}")

print("Saved:", out_path)

## GPT-4o Verification Call Block

In [ ]:
#Retrieve OpenRouter Key 
dotenv.load_dotenv(dotenv_path="PhantomDataset/.env", override=True)
API_KEY = os.getenv("OPENROUTER_API_KEY")
if not API_KEY:
    raise ValueError("Missing OPENROUTER_API_KEY in environment.")
client = genai.Client(api_key=API_KEY)

In [ ]:
out_path = "outputs/outputs/qa_generation_openai_gpt-4o.jsonl"
os.makedirs("outputs", exist_ok=True)

with open(out_path, "w") as f:
    for i, ex in enumerate(notHallu_dataset):
        q = ex["query"]
        c = ex["context"]
        gold = ex["answer"]                  # ground truth answer
        gtl = ex.get("ground_truth_label")   # your dataset label (if present)

        # If you already have a claim type label stored somewhere, pass it here.
        # If not, leave as None and the model will infer.
        claim_type = ex.get("label")  # if your enriched dataset includes it; otherwise None
        print("Now processing")
        pred = generate_qa_with_models(q, c, claim_type=claim_type, model="openai/gpt-40")

        row = {
            "idx": i,
            "query": q,
            "context": c,
            "gold_answer": gold,
            "ground_truth_label": gtl,
            "claim_type_param": claim_type,
            "claim_type_used": pred.get("claim_type_used"),
            "model_answer": pred.get("answer"),
            "model_confidence": pred.get("confidence"),
            "model_support": pred.get("support"),
        }

        f.write(json.dumps(row, ensure_ascii=False) + "\n")
        break

        if (i + 1) % 1000 == 0:
            f.flush()
            os.fsync(f.fileno())
            print(f"✅ Saved checkpoint: {i+1} rows -> {out_path}")

print("Saved:", out_path)

In [ ]:
import json 
gemini_results = []
with open("outputs/verified_gemini_output.jsonl", "r") as f:
    for line in f:
        gemini_results.append(json.loads(line))
        
print("Loaded", len(gemini_results), "rows")

In [ ]:
#Figure out the keys for model 
for key in gemini_results[0]:
    print(key)

In [ ]:


abstain_data = [a for a in gemini_results if a.get("model_answer") == "abstain"]
print(len(abstain_data))

In [ ]:
for i in range(len(abstain_data)):
    print(f"Index: {abstain_data[i]["idx"]}")
    print(abstain_data[i]["query"])
    print(abstain_data[i]["model_answer"])
    print("\n")

In [ ]:
not_abstain_dataset = [na for na in gemini_results if na.get("model_answer") != "abstain"]
print(len(not_abstain_dataset))

In [ ]:
for i in range(len(not_abstain_dataset)):
    print(f"Index: {not_abstain_dataset[i]["idx"]}")
    print(not_abstain_dataset[i]["query"])
    print(not_abstain_dataset[i]["model_answer"])

In [ ]:
print("Results")
print(f"Percentage of samples Gemini ABSTAINED from: {len(abstain_data) / len(gemini_results)}")
print(f"Percentage of samples Gemini answered: {len(not_abstain) / len(gemini_results)}")

In [ ]:
import os, time, random
from google import genai
from google.genai import errors as genai_errors

# 1) Gemini client (needs env var set)
dotenv.load_dotenv()
API_KEY = os.getenv("GEMINI_API_KEY")
if not API_KEY:
    raise ValueError("Missing GEMINI_API_KEY in environment.")
client = genai.Client(api_key=API_KEY)


QA_CONFIG = types.GenerateContentConfig(
    response_mime_type="application/json",
    temperature=0.0
)

In [ ]:
import json, os, time
from typing import Any, Dict

from google.genai import types
from google.genai import errors as genai_errors

# ---- Config for judge output ----
VERIFY_CONFIG = types.GenerateContentConfig(
    response_mime_type="application/json",
    temperature=0.0,
)

def create_prompt_verification(query: str, context: str, gold_ans: str, generated_ans: str) -> str:
    return f"""
You are an LLM judge.

Task:
Decide if the GENERATED ANSWER is HALLUCINATED or NOT_HALLUCINATED relative to the GOLD ANSWER.
Use the query and context only as supporting evidence when needed.

Rules:
- If the generated answer contradicts the gold answer, is unsupported by context, or adds specific incorrect facts: HALLUCINATED.
- If it matches the gold answer in meaning (minor wording differences ok): NOT_HALLUCINATED.
- If the generated answer is empty/None: HALLUCINATED.

Return ONLY valid JSON with this schema:
{{"verify_label":"hallucinated"|"not_hallucinated","verify_confidence":0.0,"verify_reason":"..."}}

Query:
{query}

Context:
{context}

Gold Answer:
{gold_ans}

Generated Answer:
{generated_ans}
""".strip()

def verify_genans_with_goldans(
    client,
    query: str,
    context: str,
    gold_ans: str,
    generated_ans: str,
    model: str = "gemini-2.0-flash",
) -> Dict[str, Any]:
    """
    Returns dict:
      {"verify_label": ..., "verify_confidence": ..., "verify_reason": ...}
    """
    default = {"verify_label": None, "verify_confidence": None, "verify_reason": None}

    # Handle missing generated answer deterministically
    if generated_ans is None or str(generated_ans).strip() == "":
        return {"verify_label": "hallucinated", "verify_confidence": 1.0, "verify_reason": "Generated answer is empty."}

    prompt = create_prompt_verification(query, context, gold_ans, generated_ans)

    # assumes you already have generate_with_backoff(client, ...) like your other pipeline
    resp = generate_with_backoff(client, model=model, contents=prompt, config=VERIFY_CONFIG)
    raw = resp.text.strip() if resp and resp.text else ""

    try:
        data = json.loads(raw)
        # normalize if list-wrapped
        if isinstance(data, list) and data and isinstance(data[0], dict):
            data = data[0]
        if not isinstance(data, dict):
            return default
    except Exception:
        return default

    # Minimal normalization/validationx
    label = (data.get("verify_label") or "").strip().lower()
    if label not in {"hallucinated", "not_hallucinated"}:
        label = None

    conf = data.get("verify_confidence")
    try:
        conf = float(conf) if conf is not None else None
    except Exception:
        conf = None

    reason = data.get("verify_reason")
    if isinstance(reason, str):
        reason = reason.strip()
    else:
        reason = None

    return {"verify_label": label, "verify_confidence": conf, "verify_reason": reason}


# ---------------------------
# Main loop (test 100 samples)
# ---------------------------
MAX_SAMPLES = 100
out_path = "outputs/verified_gemini_output.jsonl"
os.makedirs("outputs", exist_ok=True)

with open(out_path, "w", encoding="utf-8") as f:
    for i, ex in enumerate(not_abstain_dataset):
        if i >= MAX_SAMPLES:
            break

        q = ex["query"]
        c = ex["context"]
        gold = ex["gold_answer"]
        gtl = ex.get("ground_truth_label")
        claim_type = ex.get("label")  # or None

        # 1) Generate model answer (your existing function)
        pred = generate_qa_with_models(q, c, claim_type=claim_type, model="gemini-2.0-flash")

        model_ans = pred.get("answer")

        # 2) Verify vs gold (judge)
        verdict = verify_genans_with_goldans(
            client=client,
            query=q,
            context=c,
            gold_ans=gold,
            generated_ans=model_ans,
            model="gemini-2.0-flash",
        )

        row = {
            "idx": i,
            "query": q,
            "context": c,
            "gold_answer": gold,
            "ground_truth_label": gtl,
            "claim_type_param": claim_type,
            "claim_type_used": pred.get("claim_type_used"),
            "model_answer": model_ans,
            "model_confidence": pred.get("confidence"),
            "model_support": pred.get("support"),
            # verification fields
            "verify_label": verdict.get("verify_label"),
            "verify_confidence": verdict.get("verify_confidence"),
            "verify_reason": verdict.get("verify_reason"),
        }

        f.write(json.dumps(row, ensure_ascii=False) + "\n")

        # optional small checkpointing
        if (i + 1) % 25 == 0:
            f.flush()
            os.fsync(f.fileno())
            print(f"saved {i+1} rows")

print("Saved:", out_path)


    

In [ ]:
import json 

verifiedGeminiResults = []
with open("outputs/verified_gemini_output.jsonl", "r") as f:
    for line in f:
        verifiedGeminiResults.append(json.loads(line))
        
print(f"Loaded {len(verifiedGeminiResults)} samples")


In [ ]:
#Download library
from datasets import load_dataset

#Load Phantom dataset
dataset = load_dataset(
    "json",
    data_files="hf://datasets/obayreaps/Gemini_Classification_Phantom/qa_generation_gemini_flash.jsonl",
    )

print(f"Successfullt loaded: {len(dataset["train"])} samples. ✅")

In [ ]:
print(dataset["train"])

In [ ]:
for key in verifiedGeminiResults[0]:
    print(key)

In [ ]:
hallucinatedVerified = [x for x in verifiedGeminiResults if x["verify_label"] == "hallucinated"]

for x in hallucinatedVerified:
    print(f"{x["idx"]}")
    print(f"{x["claim_type_used"]}")
    print(f"{x["query"]}")
    print(f"Gold answer: {x["gold_answer"]}")
    print(f"Model Answer: {x["model_answer"]}")
    print(f"Label: {x["verify_label"]}")
    print("\n")

In [ ]:
print("Percentage of hallucinations within the first 100 samples are:")
print(f"{len(hallucinatedVerified) / len(verifiedGeminiResults)}")

In [ ]:
for x in verifiedGeminiResults:
    print(f"{x["idx"]}")
    print(f"{x["query"]}")
    print(f"Gold answer: {x["gold_answer"]}")
    print(f"Model Answer: {x["model_answer"]}")
    print(f"Label: {x["verify_label"]}")
    print("\n")